In [15]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

In [21]:
load_dotenv()  # Carga las variables de entorno desde el archivo .env
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')
REDIRECT_URI = os.getenv('REDIRECT_URI')

In [ ]:
# --- 1. AUTENTICACIÓN ---

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI
))

# Diccionario para traducir los números de Spotify a notas musicales
PITCH_CLASS = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

# Aquí guardaremos nuestra base de datos
dataset = {}

print("1. Obteniendo canciones de la playlist...")

playlist_id = '37i9dQZEVXbMDoHDwVN2tF'

results = sp.playlist_items(playlist_id=playlist_id)

track_ids = []
track_info_map = {}

# Extraer información básica de cada canción
for item in results['items']:
    track = item['track']
    if not track: continue
    
    t_id = track['id']
    track_ids.append(t_id)
    
    track_info_map[t_id] = {
        'name': track['name'],
        'artist': track['artists'][0]['name'],
        'popularity': track['popularity']
    }

print("2. Analizando los tonos musicales (keys)...")
# Pedir las características de audio en bloque a la API
audio_features = sp.audio_features(track_ids)

for feature in audio_features:
    if not feature: continue
    
    t_id = feature['id']
    key_num = feature['key']
    mode_num = feature['mode']
    
    # Ignorar si Spotify no logró detectar el tono (-1)
    if key_num == -1: continue
    
    # Traducir a texto (ej: "C Major" o "G Minor")
    note = PITCH_CLASS[key_num]
    mode_text = "Major" if mode_num == 1 else "Minor"
    musical_key = f"{note} {mode_text}"
    
    # Si la llave musical no existe en nuestro diccionario, la creamos
    if musical_key not in dataset:
        dataset[musical_key] = []
        
    # Añadir la canción a su respectivo tono musical
    dataset[musical_key].append(track_info_map[t_id])

print("3. Ordenando y guardando datos...")
# Ordenar las canciones dentro de cada tono por popularidad (de mayor a menor)
for k in dataset:
    dataset[k] = sorted(dataset[k], key=lambda x: x['popularity'], reverse=True)

# Guardar todo en un archivo JSON local
with open('songs_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

print("¡Éxito! Archivo 'songs_dataset.json' creado correctamente en tu carpeta.")

1. Obteniendo canciones de la playlist...


HTTP Error for GET to https://api.spotify.com/v1/playlists/37i9dQZEVXbMDoHDwVN2tF/items with Params: {'limit': 50, 'offset': 0, 'fields': None, 'market': None, 'additional_types': 'track,episode'} returned 404 due to Resource not found


SpotifyException: http status: 404, code: -1 - https://api.spotify.com/v1/playlists/37i9dQZEVXbMDoHDwVN2tF/items?limit=50&offset=0&additional_types=track%2Cepisode:
 Resource not found, reason: None